In [26]:
import pandas as pd
from datetime import datetime

t_bill_10y = pd.read_csv('data/DGS10.csv')
t_bill_2y = pd.read_csv('data/DGS2.csv')
b_a = pd.read_csv('data/BAA_AAA.csv')

t_bill_10y['DGS10'] = pd.to_numeric(t_bill_10y['DGS10'], errors='coerce')
t_bill_2y['DGS2'] = pd.to_numeric(t_bill_2y['DGS2'], errors='coerce')
b_a['BAA_AAA'] = pd.to_numeric(b_a['BAA_AAA'], errors='coerce')

for df in [t_bill_10y, t_bill_2y, b_a]:
    df['observation_date'] = pd.to_datetime(df['observation_date'])

start = pd.Timestamp('1962-01-02')
end = pd.Timestamp('2026-03-01')

t_bill_10y = t_bill_10y[(t_bill_10y['observation_date'] >= start) & (t_bill_10y['observation_date'] <= end)]
t_bill_2y  = t_bill_2y[ (t_bill_2y['observation_date']  >= start) & (t_bill_2y['observation_date']  <= end)]
b_a        = b_a[       (b_a['observation_date']        >= start) & (b_a['observation_date']        <= end)]

df = pd.merge(t_bill_10y, t_bill_2y, on='observation_date')
df = pd.merge(df, b_a, on='observation_date')

df['10Y-2Y_spread']    = df['DGS10'] - df['DGS2']
df['Premium_BBB_Spread'] = df['BAA_AAA']

df = df.dropna()

print(df.shape)
print(df)

(382, 6)
    observation_date  DGS10  DGS2  BAA_AAA  10Y-2Y_spread  Premium_BBB_Spread
0         1976-06-01   7.94  7.26     1.27           0.68                1.27
1         1976-07-01   7.88  7.02     1.26           0.86                1.26
2         1976-09-01   7.64  6.50     1.02           1.14                1.02
3         1976-10-01   7.49  6.24     0.97           1.25                0.97
4         1976-11-01   7.38  6.03     0.98           1.35                0.98
..               ...    ...   ...      ...            ...                 ...
420       2025-05-01   4.25  3.70     0.75           0.55                0.75
421       2025-07-01   4.26  3.78     0.65           0.48                0.65
422       2025-08-01   4.23  3.69     0.65           0.54                0.65
424       2025-10-01   4.12  3.55     0.61           0.57                0.61
425       2025-12-01   4.09  3.54     0.59           0.55                0.59

[382 rows x 6 columns]


In [ ]:
recession_periods = [
    ("1980-01-01", "1980-07-31"),   
    ("1981-07-01", "1982-11-30"),   
    ("1990-07-01", "1991-03-31"),   
    ("2001-03-01", "2001-11-30"),   
    ("2007-12-01", "2009-06-30"),   
    ("2020-02-01", "2020-04-30"),   
]

df["recession"] = 0
for start, end in recession_periods:
    mask = (df["observation_date"] >= start) & (df["observation_date"] <= end)
    df.loc[mask, "recession"] = 1

df['target'] = df['recession'].shift(-6)
df = df.dropna(subset=["target"])
df["target"] = df["target"].astype(int)

In [28]:
# print(df[df['recession'] == 1])

In [29]:
print(df)

    observation_date  DGS10  DGS2  BAA_AAA  10Y-2Y_spread  Premium_BBB_Spread  \
0         1976-06-01   7.94  7.26     1.27           0.68                1.27   
1         1976-07-01   7.88  7.02     1.26           0.86                1.26   
2         1976-09-01   7.64  6.50     1.02           1.14                1.02   
3         1976-10-01   7.49  6.24     0.97           1.25                0.97   
4         1976-11-01   7.38  6.03     0.98           1.35                0.98   
..               ...    ...   ...      ...            ...                 ...   
413       2024-05-01   4.63  4.96     0.70          -0.33                0.70   
414       2024-07-01   4.48  4.77     0.72          -0.29                0.72   
415       2024-08-01   3.99  4.16     0.73          -0.17                0.73   
416       2024-10-01   3.74  3.61     0.68           0.13                0.68   
417       2024-11-01   4.37  4.21     0.64           0.16                0.64   

     recession  target  
0 